# Object Detection: Exercises

This notebook accompanies the `07_object_detection` module. You will convert
box formats, compute IoU, run NMS, inspect the synthetic small-shop shelf
dataset, encode YOLO targets, check the loss gradient and train a small
detector on the CPU.

## Learning objectives

- Move between `xyxy`, `xywh` and `cxcywh` box formats without looping.
- Compute a pairwise IoU matrix and explain why zero-area boxes need care.
- Run greedy NMS and class-aware NMS.
- Generate and inspect a reproducible synthetic detection dataset.
- Encode boxes into a 6x6 YOLO grid and decode them back.
- Understand the YOLO loss: objectness BCE, coordinate error and class CE.
- Train `MiniYOLO`, decode predictions and evaluate mAP@0.5.

## 0. Setup

Import the NumPy-only implementation. The path logic lets the notebook run
either from the repository root or from this module directory.

In [ ]:
import os
import sys
import time

import numpy as np

MODULE_DIR = os.path.abspath(".")
if not os.path.exists(os.path.join(MODULE_DIR, "implementation.py")):
    MODULE_DIR = os.path.abspath("04_advanced_topics/07_object_detection")
sys.path.insert(0, MODULE_DIR)

from implementation import (
    CLASS_NAMES,
    MiniYOLO,
    cxcywh_to_xywh,
    cxcywh_to_xyxy,
    decode_single,
    decode_targets_batch,
    encode_targets,
    encode_targets_batch,
    generate_shelf_dataset,
    iou_matrix,
    mean_average_precision,
    nms,
    per_class_nms,
    xywh_to_cxcywh,
    xywh_to_xyxy,
    xyxy_to_cxcywh,
    xyxy_to_xywh,
    yolo_loss_and_grad,
)

RANDOM_STATE = 42
np.set_printoptions(precision=4, suppress=True)
print("NumPy", np.__version__)

## Exercise 1: Box formats

A detection box can be written in three common ways:

- `xyxy`: top-left and bottom-right corners,
- `xywh`: top-left corner plus width and height,
- `cxcywh`: centre plus width and height.

Convert the batch below to every format and verify that both round trips are
exact.

**Solution:** all conversion functions are vectorized. They accept `(4,)` for
one box or `(N, 4)` for a batch.

In [ ]:
boxes_xyxy = np.array([
    [4.0, 6.0, 20.0, 22.0],
    [10.0, 2.0, 18.0, 30.0],
    [0.0, 0.0, 8.0, 8.0],
])

boxes_xywh = xyxy_to_xywh(boxes_xyxy)
boxes_cxcywh = xyxy_to_cxcywh(boxes_xyxy)

print("xyxy  :", boxes_xyxy)
print("xywh  :", boxes_xywh)
print("cxcywh:", boxes_cxcywh)

assert np.allclose(xywh_to_xyxy(boxes_xywh), boxes_xyxy)
assert np.allclose(cxcywh_to_xyxy(boxes_cxcywh), boxes_xyxy)
assert np.allclose(cxcywh_to_xywh(boxes_cxcywh), boxes_xywh)
assert np.allclose(xywh_to_cxcywh(boxes_xywh), boxes_cxcywh)
print("All round trips passed.")

## Exercise 2: IoU and greedy NMS

For two boxes, IoU is intersection area divided by union area. In a detection
model we need the full pairwise matrix between `N` predictions and `M` ground
truths, so shape `(N, M)`.

NMS repeatedly keeps the highest-scoring box and removes boxes with IoU above a
threshold. Implement the idea by hand for the small batch below and compare
with `nms`.

**Solution:** `iou_matrix` broadcasts the corners; `nms` returns indices sorted
by descending score. The union can be zero for two degenerate boxes, in which
case IoU is defined as 0.

In [ ]:
a = np.array([[0.0, 0.0, 10.0, 10.0], [30.0, 30.0, 40.0, 40.0]])
b = np.array([[0.0, 0.0, 10.0, 10.0], [5.0, 0.0, 15.0, 10.0]])

print("IoU matrix:\n", iou_matrix(a, b))
assert iou_matrix(a, b)[0, 0] == 1.0
assert iou_matrix(a, b)[0, 1] == 1 / 3

scores = np.array([0.95, 0.80, 0.70])
candidate_boxes = np.array([
    [0.0, 0.0, 10.0, 10.0],
    [1.0, 1.0, 11.0, 11.0],
    [30.0, 30.0, 40.0, 40.0],
])
keep = nms(candidate_boxes, scores, iou_threshold=0.5)
print("kept indices:", keep)
assert keep.tolist() == [0, 2]

## Exercise 3: Per-class NMS

Two objects of different classes may overlap heavily. Standard NMS would
suppress one of them, so a detector applies NMS separately per class.

**Question:** what should happen for three identical boxes labelled `0, 1, 0`?

**Solution:** the two class-0 boxes suppress each other, while the class-1 box
survives. The final indices are still sorted by score.

In [ ]:
stacked = np.tile(np.array([[0.0, 0.0, 12.0, 12.0]]), (3, 1))
class_scores = np.array([0.9, 0.8, 0.7])
class_labels = np.array([0, 1, 0])

kept_classes = per_class_nms(stacked, class_scores, class_labels, 0.5)
print("kept:", kept_classes, "labels:", class_labels[kept_classes])
assert kept_classes.tolist() == [0, 1]
assert len(set(class_labels[kept_classes].tolist())) == 2

## Exercise 4: The synthetic shelf dataset

`generate_shelf_dataset` builds reproducible 48x48 images with one to four
products. The three classes have different silhouettes: cartons, bottles and
cans. Object centres are placed in distinct 6x6 cells so the one-box-per-cell
target is unambiguous.

Inspect the shapes, object counts and class balance. Verify that two calls with
the same seed are identical and that boxes do not overlap.

**Solution:** each annotation is `(boxes_xyxy, labels)` with boxes in pixel
coordinates.

In [ ]:
images, annotations = generate_shelf_dataset(n_samples=40, seed=RANDOM_STATE)
print("images:", images.shape, images.dtype)
print("first annotation:", annotations[0])

counts = np.array([len(labels) for _, labels in annotations])
all_labels = np.concatenate([labels for _, labels in annotations])
print("objects per image: min", counts.min(), "max", counts.max())
for class_id, name in enumerate(CLASS_NAMES):
    print(f"{name:>7}: {(all_labels == class_id).sum()} boxes")

images_again, annotations_again = generate_shelf_dataset(
    n_samples=40, seed=RANDOM_STATE
)
assert np.array_equal(images, images_again)
for (boxes, labels), (boxes_again, labels_again) in zip(
    annotations, annotations_again
):
    assert np.array_equal(boxes, boxes_again)
    assert np.array_equal(labels, labels_again)

for boxes, _ in annotations:
    overlaps = iou_matrix(boxes, boxes)
    np.fill_diagonal(overlaps, 0.0)
    assert overlaps.max() == 0.0
print("Reproducible and non-overlapping.")

## Exercise 5: YOLO target encoding

For `S = 6` and `C = 3`, each cell stores:

```
[objectness, x_offset, y_offset, sqrt_w, sqrt_h, class_0, class_1, class_2]
```

The object goes into the cell that contains its centre. Offsets are relative to
the cell, and the size channels are `sqrt(width / image_size)`.

Decode a batch of targets and check that every original box is recovered with
IoU 1.0.

**Solution:** `decode_targets_batch` is the inverse for in-image boxes.

In [ ]:
targets = encode_targets_batch(annotations)
print("target shape:", targets.shape)
print("max objects per image:", targets[..., 0].sum(axis=(1, 2)).max())

decoded = decode_targets_batch(targets)
for (boxes, labels), (decoded_boxes, decoded_labels) in zip(
    annotations, decoded
):
    assert len(boxes) == len(decoded_boxes)
    assert sorted(labels.tolist()) == sorted(decoded_labels.tolist())
    overlaps = iou_matrix(boxes, decoded_boxes)
    for row in range(len(boxes)):
        assert overlaps[row].max() > 0.999
print("encode -> decode round trip passed.")

single = encode_targets(
    np.array([[8.0, 8.0, 16.0, 16.0]]), np.array([1])
)
row, col = np.unravel_index(np.argmax(single[..., 0]), single.shape[:2])
print("chosen cell:", (row, col))
print("target vector:", single[row, col])
assert (row, col) == (1, 1)
assert np.isclose(single[row, col, 1], 0.5)

## Exercise 6: The YOLO loss and its gradient

The loss has four terms:

1. objectness BCE on object cells,
2. `lambda_noobj = 0.5` times objectness BCE on background cells,
3. `lambda_coord = 5` times squared error on offsets and square-root sizes,
4. softmax cross-entropy on class logits, only on object cells.

`yolo_loss_and_grad(raw, targets)` returns the loss and the gradient with
respect to the raw head output. Check it with central finite differences on a
random batch.

**Solution:** use a small step such as `1e-6` and compare the full gradient
vectors. The relative norm error should be far below `1e-5`.

In [ ]:
rng = np.random.default_rng(0)
raw = rng.normal(0.0, 0.4, size=(2, 6, 6, 8))
loss_targets = np.zeros_like(raw)
for image in range(2):
    for _ in range(3):
        row, col = rng.integers(0, 6, size=2)
        loss_targets[image, row, col, 0] = 1.0
        loss_targets[image, row, col, 1:3] = rng.random(2)
        loss_targets[image, row, col, 3:5] = rng.uniform(0.2, 0.5, size=2)
        loss_targets[image, row, col, 5 + rng.integers(0, 3)] = 1.0

loss, analytic = yolo_loss_and_grad(raw, loss_targets)
numeric = np.zeros_like(raw)
step = 1e-6
for index in np.ndindex(raw.shape):
    original = raw[index]
    raw[index] = original + step
    plus, _ = yolo_loss_and_grad(raw, loss_targets)
    raw[index] = original - step
    minus, _ = yolo_loss_and_grad(raw, loss_targets)
    raw[index] = original
    numeric[index] = (plus - minus) / (2 * step)

relative = np.linalg.norm(numeric - analytic) / (
    np.linalg.norm(numeric) + np.linalg.norm(analytic)
)
print("loss:", round(loss, 4))
print("relative gradient error:", relative)
assert relative < 1e-5

## Exercise 7: Train a small detector

Train `MiniYOLO` for 20 epochs on 36 images. Training should take a few seconds
on a laptop CPU. Then decode the held-out images and compute mAP@0.5.

**Questions:**

1. Does the loss fall substantially in the first few epochs?
2. Which class is easiest, and which is hardest?
3. Why is the mAP on only 12 test images noisy?

**Solution:** the detector usually reaches a useful mAP after a few dozen
epochs on 200+ images; this tiny run is only a smoke test.

In [ ]:
train_images, train_annotations = images[:36], annotations[:36]
test_images, test_annotations = images[36:], annotations[36:]

model = MiniYOLO(seed=5)
started = time.perf_counter()
history = model.fit(
    train_images,
    train_annotations,
    epochs=20,
    batch_size=8,
    lr=4e-3,
    seed=2,
)
elapsed = time.perf_counter() - started

print("loss history:", np.round(history, 2))
print(f"training time: {elapsed:.2f}s")
assert history[-1] < history[0]

predictions = model.predict(test_images, conf_threshold=0.1)
map_value, per_class = mean_average_precision(
    predictions, test_annotations, return_per_class=True
)
print(f"mAP@0.5: {map_value:.3f}")
for class_id, name in enumerate(CLASS_NAMES):
    print(f"  {name:>7}: AP={per_class[class_id]:.3f}")

## Exercise 8: Decode and inspect predictions

Pick one test image and print ground-truth boxes, predicted boxes, scores and
labels. Match predictions to the ground truth by IoU.

**Solution:** `decode_single` applies `sigmoid(objectness) * max(class_prob)`,
thresholds and then per-class NMS.

In [ ]:
sample = 0
image = test_images[sample]
gt_boxes, gt_labels = test_annotations[sample]
pred_boxes, pred_scores, pred_labels = predictions[sample]

print("ground truth:")
for box, label in zip(gt_boxes, gt_labels):
    print(f"  {CLASS_NAMES[int(label)]:>7} {np.round(box, 1)}")
print("predictions:")
for box, score, label in zip(pred_boxes, pred_scores, pred_labels):
    print(f"  {CLASS_NAMES[int(label)]:>7} {np.round(box, 1)} score={score:.3f}")

if len(pred_boxes):
    overlaps = iou_matrix(gt_boxes, pred_boxes)
    print("best IoU per GT:", np.round(overlaps.max(axis=1), 2))
else:
    print("no prediction passed the confidence threshold")

high_confidence_boxes, _, _ = decode_single(
    model.forward(image[None])[0], conf_threshold=0.9
)
print(
    "high-confidence decoding keeps",
    len(high_confidence_boxes),
    "of",
    len(pred_boxes),
    "boxes.",
)
assert len(high_confidence_boxes) <= len(pred_boxes)

## Exercise 9: Challenge -- confidence threshold sweep

Vary the confidence threshold and compute mAP for each value. Then discuss:

1. Which threshold would you pick on a validation set, not on the test set?
2. Why can a lower threshold help mAP even when it adds false positives?
3. What changes would a real detector need for small, occluded or rotated
   products?

**Solution:** mAP rewards ranking quality and recall, not just precision. A
detector that is too conservative loses recall on hard objects. Select the
threshold on a validation split and report the final number once on the test
split.

In [ ]:
rows = []
for threshold in (0.05, 0.10, 0.20, 0.40, 0.60):
    threshold_predictions = model.predict(
        test_images, conf_threshold=threshold
    )
    threshold_map = mean_average_precision(
        threshold_predictions, test_annotations
    )
    rows.append((threshold, threshold_map))
    print(f"threshold={threshold:.2f}  mAP@0.5={threshold_map:.3f}")

best_threshold = max(rows, key=lambda row: row[1])
print("best threshold in this sweep:", best_threshold[0])
print("\nRemember: this is a synthetic teaching dataset, not a production benchmark.")